# HuBERT — Self-Supervised Speech via Masked Prediction of Hidden Units (2021)

_Papers · Speech & Audio_

**Learn speech representations with no labels by clustering audio features into pseudo-labels (k-means "hidden units"), then BERT-masking spans and predicting those cluster IDs — refined over a few iterations.**

---

This notebook is a practice scaffold for the **HuBERT — Self-Supervised Speech via Masked Prediction of Hidden Units (2021)** lesson. We assemble a tiny HuBERT one stage at a time: the masked-frame loss, synthetic "speech" frames, k-means pseudo-labels, span masking, a small bidirectional encoder, and finally the masked-prediction training plus an ablation. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — The masked-frame loss, then synthetic "speech"

First we check HuBERT's core objective on a single masked frame: cross-entropy over `C=4` clusters. The logits are codeword cosine-similarities divided by a temperature (Eq. 3); the loss is just $-\log p(\text{true cluster})$, which we confirm matches `F.cross_entropy`. Then we fabricate toy "speech": six sequences of 24 frames, where each frame is a noisy 2-D point drawn near one of four latent sound "units", with units repeating in short runs the way real phones last several frames.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# === 0. Worked example: cross-entropy at ONE masked frame over C=4 clusters. ===
# Logits are the codeword cosine-similarities / temperature (Eq 3). True cluster = index 2.
logits = torch.tensor([1.0, 3.0, 0.5, -1.0])
target = 2

p = F.softmax(logits, dim=0)
print("softmax cluster probs =", [round(v, 4) for v in p.tolist()])   # [0.1095, 0.8093, 0.0664, 0.0148]
print("p(true cluster 2)     =", round(p[target].item(), 4))          # 0.8093
print("masked loss = -log p  =", round(-math.log(p[target].item()), 4))           # 0.2115
print("F.cross_entropy       =", round(F.cross_entropy(logits.unsqueeze(0), torch.tensor([target])).item(), 4))  # 0.2115

# === 1. Toy "speech": 6 sequences of 24 frames; each frame is a 2-D feature drawn near one of C latent units. ===
C, SEQ, N, FEAT = 4, 24, 6, 2
centres_true = torch.tensor([[2.,2.], [-2.,2.], [-2.,-2.], [2.,-2.]])   # 4 hidden "sound units"

# build sequences with locally-coherent runs of the same unit (like real phones lasting several frames)
def make_seq():
    ids, t = [], 0
    while t < SEQ:
        u = torch.randint(0, C, (1,)).item()
        run = int(torch.randint(2, 6, (1,)).item())
        ids += [u] * run
        t += run
    return torch.tensor(ids[:SEQ])

true_ids = torch.stack([make_seq() for _ in range(N)])                  # (N, SEQ) latent units (unknown to model)
feats = centres_true[true_ids] + 0.45 * torch.randn(N, SEQ, FEAT)       # (N, SEQ, 2) noisy frame features

### Step 2 — Discover pseudo-labels (k-means) and pick spans to mask

HuBERT has no transcripts, so it *invents* targets: run k-means on the frame features to assign each frame a cluster id $z_t$ (acoustic-unit discovery, §II-A). The cluster ids are arbitrary, but we print their agreement with the hidden units just to show the clusters track real structure. Then we set up span masking (§II-B): mark ~8% of frames as span *starts* and mask a run of length 10 from each, mimicking BERT-style masked spans.

In [ ]:
# === 2. Acoustic unit discovery = k-means on the frame features -> PSEUDO-LABELS z_t (\S II-A). ===
def kmeans(x, k, iters=25):
    x = x.reshape(-1, x.shape[-1])
    cent = x[torch.randperm(x.shape[0])[:k]].clone()
    for _ in range(iters):
        d = torch.cdist(x, cent)                                        # frame-to-centre distances
        a = d.argmin(1)                                                 # nearest centre = cluster id
        for c in range(k):
            if (a == c).any():
                cent[c] = x[a == c].mean(0)
    return a, cent

labels, _ = kmeans(feats, C)
labels = labels.reshape(N, SEQ)                                         # (N, SEQ) pseudo-labels z_t in [C]
print("\npseudo-label purity vs latent units:",
      round((labels.reshape(-1) == true_ids.reshape(-1)).float().mean().item(), 3),
      "(k-means ids are arbitrary; this is just to show clusters track the hidden units)")

# === 3. Span masking: 8% of frames as span STARTS, span length l=10 (\S II-B). ===
MASK_LEN = 10
def mask_spans(B, p=0.08, l=MASK_LEN):
    m = torch.zeros(B, SEQ, dtype=torch.bool)
    starts = torch.rand(B, SEQ) < p
    for b in range(B):
        idx = starts[b].nonzero().flatten().tolist()
        if not idx:
            idx = [int(torch.randint(0, SEQ, (1,)))]                    # tiny data: guarantee >=1 span
        for s in idx:
            m[b, s:min(s + l, SEQ)] = True
    return m

### Step 3 — A tiny bidirectional encoder with a cluster-prediction head

The model is a miniature Transformer encoder. Multi-head attention uses **no causal mask**, so every frame attends to frames on both sides — that bidirectionality is what lets it infer a masked frame from surrounding context. Masked frames have their projected features overwritten by a learned `[MASK]` vector before positional encodings are added; after the encoder blocks, a linear head emits one logit per cluster for every frame.

In [ ]:
# === 4. Tiny bidirectional encoder (no causal mask) + cluster-prediction head. ===
class MHA(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h, self.dk = h, d // h
        self.Wq, self.Wk, self.Wv, self.Wo = (nn.Linear(d, d) for _ in range(4))
    def split(self, x):
        B, S, _ = x.shape
        return x.view(B, S, self.h, self.dk).transpose(1, 2)
    def forward(self, x):
        Q  = self.split(self.Wq(x))
        K  = self.split(self.Wk(x))
        Vv = self.split(self.Wv(x))
        a = F.softmax(Q @ K.transpose(-2, -1) / math.sqrt(self.dk), dim=-1) @ Vv  # no causal mask -> bidirectional
        B, _, S, _ = a.shape
        return self.Wo(a.transpose(1, 2).contiguous().view(B, S, self.h * self.dk))

class Block(nn.Module):
    def __init__(self, d, h, ff):
        super().__init__()
        self.a = MHA(d, h)
        self.f = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Linear(ff, d))
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
    def forward(self, x):
        x = self.n1(x + self.a(x))
        return self.n2(x + self.f(x))

def positional_encoding(seq_len, d_model):
    pos = torch.arange(seq_len).unsqueeze(1).float()
    i2 = torch.arange(0, d_model, 2).float()
    denom = torch.pow(10000.0, i2 / d_model)
    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(pos / denom)
    pe[:, 1::2] = torch.cos(pos / denom)
    return pe

class TinyHuBERT(nn.Module):
    def __init__(self, C, feat=FEAT, d=64, h=4, ff=128, L=2, mx=SEQ):
        super().__init__()
        self.proj = nn.Linear(feat, d)                                 # project raw frame features in
        self.mask_emb = nn.Parameter(torch.randn(d))                   # the learned [MASK] vector
        self.register_buffer("pe", positional_encoding(mx, d))
        self.b = nn.ModuleList([Block(d, h, ff) for _ in range(L)])
        self.head = nn.Linear(d, C)                                    # cluster-prediction head: logit per cluster
    def forward(self, feats, mask):
        x = self.proj(feats)
        x = torch.where(mask.unsqueeze(-1), self.mask_emb, x)          # overwrite masked frames with [MASK]
        x = x + self.pe[:x.shape[1]]
        for blk in self.b:
            x = blk(x)
        return self.head(x)                                            # (B, SEQ, C)

### Step 4 — Train on masked frames, then ablate the masking

The loss is $L = \alpha L_m + (1-\alpha) L_u$ (§II-B): $\alpha$ weights the loss on **masked** frames, $1-\alpha$ on **unmasked** ones. We first train with $\alpha=1$ (masked-only) and watch masked-frame cluster accuracy climb. Then the ablation with $\alpha=0$ scores only unmasked frames, so the model is never forced to infer a hidden frame from context — and its masked-frame accuracy collapses toward chance ($1/C = 0.25$), mirroring the paper's Table V.

In [ ]:
# === 5. Train with the masked-prediction loss L = alpha*L_m + (1-alpha)*L_u (\S II-B). alpha=1 -> masked only. ===
def train(alpha=1.0, steps=600, lr=3e-3, seed=0):
    torch.manual_seed(seed)
    net = TinyHuBERT(C)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    hist = []
    for s in range(steps):
        m = mask_spans(N)
        logit = net(feats, m)                                          # (N, SEQ, C)
        lm = F.cross_entropy(logit[m], labels[m])                      # loss on MASKED frames
        lu = F.cross_entropy(logit[~m], labels[~m])                    # loss on UNMASKED frames
        loss = alpha * lm + (1 - alpha) * lu
        opt.zero_grad()
        loss.backward()
        opt.step()
        # track accuracy on MASKED frames (the quantity that matters)
        with torch.no_grad():
            acc = (logit[m].argmax(-1) == labels[m]).float().mean().item()
        hist.append((lm.item(), acc))
        if s % 100 == 0 or s == steps - 1:
            print(f"  step {s:4d}  masked-loss {lm.item():.4f}  masked-acc {acc:.3f}")
    return net, hist

print("\nTRAIN tiny HuBERT (alpha=1, masked-prediction only):")
net, hist = train(alpha=1.0)

# === 6. ABLATION: alpha=0 -> score loss only on UNMASKED frames (Table V). Masked-frame accuracy collapses. ===
print("\nABLATION (alpha=0, unmasked-only) -- the model is never graded on hidden frames:")
_, hist0 = train(alpha=0.0)
print(f"\nfinal masked-frame accuracy  alpha=1 (masked-only): {hist[-1][1]:.3f}   alpha=0 (unmasked-only): {hist0[-1][1]:.3f}")
# alpha=1 learns to fill masked frames from context; alpha=0 never does (near chance = 1/C = 0.25).
# Our small synthetic run, not the paper's WER (Table V reports 17.86% vs 96.37% WER).

## Visualize the data & results

_Does the masked-prediction loss fall (and masked-frame cluster accuracy rise) as we train a tiny HuBERT with alpha=1 (masked-only), and does the alpha=0 (unmasked-only) ablation fail to predict masked frames?_

### Step 1 — Re-declare the world and the model (self-contained)

This visualization panel stands on its own, so it re-imports torch and re-declares the same toy-speech constants, the `make_seq` / `kmeans` / `mask_spans` helpers, the positional encoding, and the `MHA` / `Block` / `TinyHuBERT` modules. Nothing here is new logic — it is the exact same building blocks from the reference cell, gathered so this section runs independently.

In [ ]:
import math, torch, torch.nn as nn, torch.nn.functional as F

C, SEQ, N, FEAT = 4, 24, 6, 2
centres_true = torch.tensor([[2.,2.], [-2.,2.], [-2.,-2.], [2.,-2.]])

def make_seq():
    ids, t = [], 0
    while t < SEQ:
        u = torch.randint(0, C, (1,)).item(); run = int(torch.randint(2, 6, (1,)).item())
        ids += [u] * run; t += run
    return torch.tensor(ids[:SEQ])

def kmeans(x, k, iters=25):
    x = x.reshape(-1, x.shape[-1]); cent = x[torch.randperm(x.shape[0])[:k]].clone()
    for _ in range(iters):
        a = torch.cdist(x, cent).argmin(1)
        for c in range(k):
            if (a == c).any(): cent[c] = x[a == c].mean(0)
    return a

def mask_spans(B, p=0.08, l=10):
    m = torch.zeros(B, SEQ, dtype=torch.bool); starts = torch.rand(B, SEQ) < p
    for b in range(B):
        idx = starts[b].nonzero().flatten().tolist()
        if not idx: idx = [int(torch.randint(0, SEQ, (1,)))]
        for s in idx: m[b, s:min(s + l, SEQ)] = True
    return m

def positional_encoding(seq_len, d_model):
    pos = torch.arange(seq_len).unsqueeze(1).float(); i2 = torch.arange(0, d_model, 2).float()
    denom = torch.pow(10000.0, i2 / d_model); pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(pos / denom); pe[:, 1::2] = torch.cos(pos / denom); return pe

class MHA(nn.Module):
    def __init__(self, d, h):
        super().__init__(); self.h, self.dk = h, d // h
        self.Wq, self.Wk, self.Wv, self.Wo = (nn.Linear(d, d) for _ in range(4))
    def split(self, x):
        B, S, _ = x.shape; return x.view(B, S, self.h, self.dk).transpose(1, 2)
    def forward(self, x):
        Q, K, Vv = self.split(self.Wq(x)), self.split(self.Wk(x)), self.split(self.Wv(x))
        a = F.softmax(Q @ K.transpose(-2, -1) / math.sqrt(self.dk), dim=-1) @ Vv
        B, _, S, _ = a.shape
        return self.Wo(a.transpose(1, 2).contiguous().view(B, S, self.h * self.dk))

class Block(nn.Module):
    def __init__(self, d, h, ff):
        super().__init__(); self.a = MHA(d, h)
        self.f = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Linear(ff, d))
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
    def forward(self, x):
        x = self.n1(x + self.a(x)); return self.n2(x + self.f(x))

class TinyHuBERT(nn.Module):
    def __init__(self, C, feat=FEAT, d=64, h=4, ff=128, L=2, mx=SEQ):
        super().__init__(); self.proj = nn.Linear(feat, d)
        self.mask_emb = nn.Parameter(torch.randn(d))
        self.register_buffer("pe", positional_encoding(mx, d))
        self.b = nn.ModuleList([Block(d, h, ff) for _ in range(L)]); self.head = nn.Linear(d, C)
    def forward(self, feats, mask):
        x = self.proj(feats); x = torch.where(mask.unsqueeze(-1), self.mask_emb, x)
        x = x + self.pe[:x.shape[1]]
        for blk in self.b: x = blk(x)
        return self.head(x)

### Step 2 — One training run that returns the masked-accuracy curve

`run(alpha)` does a full self-contained experiment: regenerate the toy speech, k-means the pseudo-labels, build a fresh `TinyHuBERT`, and train for 600 steps recording masked-frame accuracy at every step. It is the same loop as the reference cell, just packaged to hand back the accuracy history so we can average it.

In [ ]:
def run(alpha, steps=600, seed=0):
    torch.manual_seed(seed)
    true_ids = torch.stack([make_seq() for _ in range(N)])
    feats = centres_true[true_ids] + 0.45 * torch.randn(N, SEQ, FEAT)
    labels = kmeans(feats, C).reshape(N, SEQ)
    net = TinyHuBERT(C); opt = torch.optim.Adam(net.parameters(), lr=3e-3); acc_hist = []
    for s in range(steps):
        m = mask_spans(N); logit = net(feats, m)
        lm = F.cross_entropy(logit[m], labels[m]); lu = F.cross_entropy(logit[~m], labels[~m])
        loss = alpha * lm + (1 - alpha) * lu
        opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad():
            acc_hist.append((logit[m].argmax(-1) == labels[m]).float().mean().item())
    return acc_hist

### Step 3 — Average over seeds and compare $\alpha=1$ vs $\alpha=0$

To get a stable picture we average the accuracy curves over four seeds for each setting, then print a few checkpoints. The masked-only run ($\alpha=1$) should climb toward ~0.97 masked-frame accuracy, while the unmasked-only ablation ($\alpha=0$) hovers near chance (~0.25 = $1/C$) — the visual version of the contrast from Step 4 above.

In [ ]:
import statistics
def avg(alpha, seeds=(0, 1, 2, 3)):
    return [statistics.mean(col) for col in zip(*[run(alpha, seed=s) for s in seeds])]

a1, a0 = avg(1.0), avg(0.0)
idx = list(range(0, 600, 50)) + [599]
print("alpha=1 (masked-only):", [[i, round(a1[i], 3)] for i in idx])
print("alpha=0 (unmasked-only):", [[i, round(a0[i], 3)] for i in idx])
# alpha=1 -> ~0.97 masked-frame accuracy; alpha=0 -> ~0.25 (chance, 1/C). Our small run, not the paper's WER.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation (Table V). Train the tiny HuBERT two ways on the same pseudo-labels: $\alpha=1$
            (cross-entropy only on masked frames) versus $\alpha=0$ (cross-entropy only on unmasked
            frames). Each time, evaluate how well it predicts the cluster ID at masked frames. What
            happens, and what does it show about why HuBERT masks?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Change exactly one thing: $\alpha$. Keep the pseudo-labels, the 8%/length-10 span masking, depth, width, heads, optimizer, and seed identical. — _An honest ablation isolates the variable under test &mdash; masked-only vs unmasked-only &mdash; so any difference is attributable to it._
- Evaluate masked-frame cluster accuracy for each. In our run $\alpha=1$ reaches high masked accuracy while $\alpha=0$ stays near chance on masked frames. — _With $\alpha=0$ the model is only ever graded on frames whose content it can already see, so it never learns to infer a hidden frame from context._
- Conclude that the masked objective &mdash; not extra capacity &mdash; is what teaches context. — _Both runs share architecture, labels, and seed; only which frames are scored differs, isolating masking as the cause (mirrors Table V: 17.86% vs 96.37% WER)._

**Answer:** With $\alpha=0$ (loss only on unmasked frames) the model can read each scored frame's
                 own content and predict its cluster almost trivially &mdash; so it never learns to fill a hidden
                 frame, and its accuracy on masked frames stays near chance. With $\alpha=1$ (masked-only)
                 the scored frames have their content deleted, forcing the model to use both-sided context, and
                 masked-frame accuracy climbs. This mirrors Table V, where the weak-teacher WER is "17.86%" at
                 $\alpha=1$ but collapses to "96.37%" at $\alpha=0$: masking is what makes the noisy
                 cluster labels usable. The CODEVIZ panel shows both curves.

</details>

**Problem 2.** In the worked example the four cluster logits at a masked frame were $[1.0, 3.0, 0.5, -1.0]$ and the
            true cluster was index 2, giving loss $0.2115$. Suppose the model were instead certain and wrong
            &mdash; a huge logit on cluster 4 and small logits elsewhere. Would the loss be larger or smaller, and
            why does that push the model in the right direction?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall the loss reads only the true cluster's probability: $-\log p_f(z_t\mid\tilde X,t)$ with $z_t=2$. — _Cross-entropy ignores how mass splits among the wrong clusters; only the probability on the true cluster matters._
- A huge logit on the wrong cluster (4) drains probability from cluster 2, so $p_f(2)$ becomes tiny, and $-\log$ of a tiny number is large. — _Confident-and-wrong is exactly the case cross-entropy punishes hardest._
- Gradient descent on that large loss raises the score (codeword similarity) for the true cluster and lowers the others next time. — _That is how masked prediction teaches the encoder to map a masked frame's context to its real cluster._

**Answer:** Larger. The loss depends only on the probability the model gives the true cluster
                 (index 2). Pouring confidence onto cluster 4 starves index 2, so $p_f(2)$ is tiny and
                 $-\log p_f(2)$ is big &mdash; far above $0.2115$. The large gradient then pushes the
                 cluster-prediction head (Eq 3) to raise the true cluster's codeword similarity and lower the
                 others, which is precisely how masked prediction trains useful speech representations.

</details>

**Problem 3.** HuBERT clusters with k-means to make the labels, even though k-means on raw MFCC features is
            crude and the clusters are not real phonemes. Why is a noisy, made-up target good enough &mdash; and
            why does the paper iterate (re-cluster on the model's own features) instead of clustering once?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note the target only needs to be a consistent function of sound: similar frames get the same cluster. — _Masked prediction then learns the regularities the labels agree on; random labelling errors don't correlate with context, so they average out (\S II-A)._
- Recognise that a model trained on crude targets still extracts better features than raw MFCC. — _Its internal layers capture phonetic structure even though the supervision was noisy &mdash; a weak teacher yields a stronger student._
- Re-cluster on those better features to get cleaner targets, then pre-train again. — _Iteration 2 clusters layer-6 features into 500 units (\S IV-B); the fixed point is a far cleaner unit inventory than the MFCC start._

**Answer:** The target need not be "correct" &mdash; there is no phoneme labelling available. It only has to
                 be consistent: similar sounds get the same cluster ID. Because the model predicts a masked
                 frame from context, it learns the structure the labels agree on and averages out their random
                 errors, so noisy k-means labels still teach good representations. And since the resulting model's
                 internal features are better than raw MFCC, re-clustering on them (iteration 2: 500
                 clusters on layer 6, &sect;IV-B) yields cleaner targets &mdash; a bootstrap that improves with
                 each round, which is why HuBERT iterates rather than clustering once.

</details>